# Ingestão e diagnóstico dos dados — Camada Bronze

Este notebook carrega oito arquivos CSV do conjunto de dados Olist
e os salva como tabelas Delta no esquema `olist_mvp.bronze`.
Essas tabelas constituem a base para o tratamento na Silver
e para as análises da Gold.

## Dados utilizados

São utilizados dados de clientes, pedidos, itens dos pedidos,
pagamentos, avaliações, produtos, vendedores e traduções de
categorias. O arquivo de geolocalização não integra esta carga,
pois as análises propostas não utilizam coordenadas geográficas.

Os arquivos de origem estão armazenados no volume
`/Volumes/olist_mvp/bronze/raw_files`.

## Critérios de ingestão

As colunas dos arquivos são lidas como texto, sem inferência
automática de tipos. A conversão de datas e números ocorre
na Silver, permitindo consultar na Bronze os valores anteriores
ao tratamento.

A leitura considera cabeçalhos, codificação UTF-8 e campos
entre aspas que podem conter quebras de linha, como os comentários
das avaliações.

São acrescentados dois metadados:
- `_source_file`: nome do arquivo de origem;
- `_ingestion_timestamp`: momento da ingestão.

A carga substitui as tabelas de destino a cada execução.
Não são aplicadas correções, preenchimentos ou exclusões
de registros nesta etapa.

## Diagnóstico inicial

Após a carga, são avaliados o volume de registros, as duplicatas
exatas, os valores ausentes por coluna, a completude e a unicidade
das chaves candidatas e as referências entre tabelas.

Cada resultado é acompanhado de interpretação para orientar
o tratamento e evitar exclusões indevidas nas etapas seguintes.

In [0]:
from pyspark.sql import functions as F

CAMINHO_ARQUIVOS_BRUTOS = "/Volumes/olist_mvp/bronze/raw_files"

arquivos_tabelas = {
    "olist_customers_dataset.csv": "customers",
    "olist_orders_dataset.csv": "orders",
    "olist_order_items_dataset.csv": "order_items",
    "olist_order_payments_dataset.csv": "payments",
    "olist_order_reviews_dataset.csv": "reviews",
    "olist_products_dataset.csv": "products",
    "olist_sellers_dataset.csv": "sellers",
    "product_category_name_translation.csv": "category_translation",
}

arquivos_disponiveis = {
    arquivo.name
    for arquivo in dbutils.fs.ls(CAMINHO_ARQUIVOS_BRUTOS)
}

arquivos_ausentes = set(arquivos_tabelas) - arquivos_disponiveis

if arquivos_ausentes:
    raise ValueError(
        f"Arquivos ausentes: {sorted(arquivos_ausentes)}"
    )

print(
    f"Conferência concluída: os {len(arquivos_tabelas)} "
    "arquivos previstos estão disponíveis para ingestão."
)

Conferência concluída: os 8 arquivos previstos estão disponíveis para ingestão.


In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS olist_mvp.bronze")

resumo_ingestao = []

for nome_arquivo, nome_tabela in arquivos_tabelas.items():
    tabela_destino = f"olist_mvp.bronze.{nome_tabela}"

    dados = (
        spark.read
        .option("header", True)
        .option("inferSchema", False)
        .option("encoding", "UTF-8")
        .option("multiLine", True)
        .option("quote", '"')
        .option("escape", '"')
        .option("mode", "FAILFAST")
        .csv(f"{CAMINHO_ARQUIVOS_BRUTOS}/{nome_arquivo}")
        .withColumn("_source_file", F.lit(nome_arquivo))
        .withColumn("_ingestion_timestamp", F.current_timestamp())
    )

    (
        dados.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(tabela_destino)
    )

    resumo_ingestao.append((
        nome_arquivo,
        tabela_destino,
        spark.table(tabela_destino).count()
    ))

display(
    spark.createDataFrame(
        resumo_ingestao,
        "arquivo_origem string, tabela_destino string, linhas long"
    )
)

arquivo_origem,tabela_destino,linhas
olist_customers_dataset.csv,olist_mvp.bronze.customers,99441
olist_orders_dataset.csv,olist_mvp.bronze.orders,99441
olist_order_items_dataset.csv,olist_mvp.bronze.order_items,112650
olist_order_payments_dataset.csv,olist_mvp.bronze.payments,103886
olist_order_reviews_dataset.csv,olist_mvp.bronze.reviews,99224
olist_products_dataset.csv,olist_mvp.bronze.products,32951
olist_sellers_dataset.csv,olist_mvp.bronze.sellers,3095
product_category_name_translation.csv,olist_mvp.bronze.category_translation,71


## Resultado da ingestão

Foram salvas oito tabelas Delta na camada Bronze, incluindo
99.441 pedidos, 112.650 itens e 103.886 registros de pagamento.

As tabelas possuem diferentes unidades de registro. Um pedido
pode conter vários itens e registros de pagamento; portanto,
essas contagens não devem ser interpretadas como quantidades
equivalentes de vendas.

A tabela de clientes contém 99.441 registros, mas essa quantidade
não representa necessariamente pessoas distintas. A identificação
de clientes recorrentes depende do campo `customer_unique_id`.

Os metadados acrescentados permitem identificar o arquivo de origem
e o momento da carga. As colunas recebidas permanecem como texto
para o tratamento posterior na Silver.

A conclusão da carga confirma a disponibilidade das tabelas,
mas não garante a qualidade de seu conteúdo. A próxima etapa
verifica duplicatas exatas e o preenchimento dos campos.

In [0]:
from pyspark.sql import functions as F

tabelas = list(arquivos_tabelas.values())

resumo_duplicatas = []

for tabela in tabelas:
    # Compara apenas as colunas recebidas da fonte.
    dados = (
        spark.table(f"olist_mvp.bronze.{tabela}")
        .drop("_source_file", "_ingestion_timestamp")
    )

    total_linhas = dados.count()
    linhas_distintas = dados.dropDuplicates().count()

    resumo_duplicatas.append((
        tabela,
        total_linhas,
        len(dados.columns),
        total_linhas - linhas_distintas
    ))

display(
    spark.createDataFrame(
        resumo_duplicatas,
        """
        tabela string,
        linhas long,
        colunas_origem long,
        duplicatas_exatas long
        """
    ).orderBy("tabela")
)

tabela,linhas,colunas_origem,duplicatas_exatas
category_translation,71,2,0
customers,99441,5,0
order_items,112650,7,0
orders,99441,8,0
payments,103886,5,0
products,32951,9,0
reviews,99224,7,0
sellers,3095,4,0


## Duplicatas exatas

Não foram identificadas linhas inteiramente repetidas nas oito
tabelas, considerando todas as colunas de origem e desconsiderando
os metadados de ingestão. Portanto, não há registros a remover
por esse critério.

Esse resultado não garante a unicidade dos identificadores.
Duas linhas podem compartilhar uma chave e apresentar diferenças
em outros campos. A unicidade das chaves será examinada separadamente.

A próxima verificação avalia o preenchimento de cada coluna,
considerando como ausentes os valores nulos, os textos vazios
e os textos compostos apenas por espaços.

In [0]:
from pyspark.sql import functions as F

def campo_ausente(coluna):
    return (
        F.col(coluna).isNull()
        | (F.trim(F.col(coluna)) == "")
    )

resultados_completude = []

for tabela in tabelas:
    dados = (
        spark.table(f"olist_mvp.bronze.{tabela}")
        .drop("_source_file", "_ingestion_timestamp")
    )

    contagens = dados.agg(
        F.count("*").alias("_total_linhas"),
        *[
            F.coalesce(
                F.sum(
                    F.when(campo_ausente(coluna), 1).otherwise(0)
                ),
                F.lit(0)
            ).alias(coluna)
            for coluna in dados.columns
        ]
    ).first()

    total = contagens["_total_linhas"]

    for coluna in dados.columns:
        ausentes = contagens[coluna]

        resultados_completude.append((
            tabela,
            coluna,
            total,
            total - ausentes,
            ausentes,
            round(100 * ausentes / total, 2) if total else None
        ))

display(
    spark.createDataFrame(
        resultados_completude,
        """
        tabela string,
        coluna string,
        total_linhas long,
        preenchidos long,
        ausentes long,
        percentual_ausentes double
        """
    ).orderBy(F.desc("percentual_ausentes"), "tabela", "coluna")
)

tabela,coluna,total_linhas,preenchidos,ausentes,percentual_ausentes
reviews,review_comment_title,99224,11566,87658,88.34
reviews,review_comment_message,99224,40968,58256,58.71
orders,order_delivered_customer_date,99441,96476,2965,2.98
products,product_category_name,32951,32341,610,1.85
products,product_description_lenght,32951,32341,610,1.85
products,product_name_lenght,32951,32341,610,1.85
products,product_photos_qty,32951,32341,610,1.85
orders,order_delivered_carrier_date,99441,97658,1783,1.79
orders,order_approved_at,99441,99281,160,0.16
products,product_height_cm,32951,32949,2,0.01


## Completude dos campos

Das 47 colunas de origem analisadas, 34 não apresentaram valores
ausentes segundo o critério adotado.

Nas avaliações, 88,34% dos títulos e 58,71% dos comentários
estão ausentes. Entretanto, todas as notas estão preenchidas.
Assim, a ausência de texto não impede a análise das notas,
mas limita análises baseadas no conteúdo dos comentários.

Nos pedidos, faltam datas de entrega ao cliente em 2,98%
dos registros, de envio em 1,79% e de aprovação em 0,16%.
Essas ausências precisam ser interpretadas em conjunto com
a situação do pedido: uma etapa não realizada pode não possuir
data registrada. A consistência entre datas e situações
é examinada na Silver.

Nos produtos, a categoria, os comprimentos do nome e da descrição
e a quantidade de fotos apresentam 610 valores ausentes por coluna
(1,85%). Contagens iguais não comprovam, isoladamente, que as
ausências ocorram nos mesmos produtos. Peso e cada uma das três
dimensões apresentam duas ausências.

As tabelas de clientes, itens, pagamentos, vendedores e traduções
de categorias não apresentaram ausências em suas colunas.
Isso indica completude, mas não garante validade ou exatidão.

Nenhum registro foi excluído por falta de preenchimento.
Na Silver, valores desconhecidos permanecem nulos; na Gold,
a elegibilidade depende dos campos necessários a cada indicador.
Produtos sem categoria podem ser apresentados como “Não informada”,
preservando sua contribuição para os totais de vendas.

In [0]:
from functools import reduce
from pyspark.sql import functions as F

verificacoes_chaves = [
    ("customers", ["customer_id"]),
    ("orders", ["order_id"]),
    ("order_items", ["order_id", "order_item_id"]),
    ("payments", ["order_id", "payment_sequential"]),
    ("products", ["product_id"]),
    ("sellers", ["seller_id"]),
    ("category_translation", ["product_category_name"]),
    ("reviews", ["review_id"]),
    ("reviews", ["review_id", "order_id"]),
]

resultados_chaves = []

for tabela, chaves in verificacoes_chaves:
    dados = spark.table(f"olist_mvp.bronze.{tabela}")

    chave_ausente = reduce(
        lambda acumulado, condicao: acumulado | condicao,
        [campo_ausente(coluna) for coluna in chaves]
    )

    # Repetições são verificadas apenas entre chaves completas.
    repeticoes = (
        dados.filter(~chave_ausente)
        .groupBy(*chaves)
        .count()
        .filter(F.col("count") > 1)
    )

    contagens = repeticoes.agg(
        F.count("*").alias("grupos_repetidos"),
        F.coalesce(
            F.sum(F.col("count") - 1),
            F.lit(0)
        ).alias("linhas_excedentes")
    ).first()

    resultados_chaves.append((
        tabela,
        " + ".join(chaves),
        dados.filter(chave_ausente).count(),
        contagens["grupos_repetidos"],
        contagens["linhas_excedentes"]
    ))

display(
    spark.createDataFrame(
        resultados_chaves,
        """
        tabela string,
        chave_testada string,
        linhas_com_chave_ausente long,
        grupos_com_chave_repetida long,
        linhas_excedentes long
        """
    )
)

tabela,chave_testada,linhas_com_chave_ausente,grupos_com_chave_repetida,linhas_excedentes
customers,customer_id,0,0,0
orders,order_id,0,0,0
order_items,order_id + order_item_id,0,0,0
payments,order_id + payment_sequential,0,0,0
products,product_id,0,0,0
sellers,seller_id,0,0,0
category_translation,product_category_name,0,0,0
reviews,review_id,0,789,814
reviews,review_id + order_id,0,0,0


## Completude e unicidade das chaves

Nenhuma das chaves testadas apresentou valores ausentes.
Nas sete tabelas além de avaliações, as chaves simples ou
compostas verificadas também não apresentaram repetições.

Na tabela `reviews`, o identificador `review_id` apresentou
789 grupos repetidos, com 814 linhas além da primeira ocorrência
de cada identificador. Essas linhas não são duplicatas exatas
e não devem ser excluídas apenas pela repetição desse campo.

A combinação `review_id` e `order_id` não apresentou repetições,
sendo uma chave candidata no conjunto analisado. Esse resultado
não implica que exista apenas uma avaliação por pedido, pois
um mesmo pedido pode estar associado a diferentes identificadores
de avaliação.

Por esse motivo, a análise da Gold agrega as notas válidas por
pedido antes de associá-las às entregas, evitando atribuir maior
peso aos pedidos com mais avaliações.

A unicidade observada descreve os dados desta carga; ela não
constitui uma restrição automaticamente aplicada às tabelas.
A próxima verificação examina as referências entre tabelas.

In [0]:
from pyspark.sql import functions as F

relacoes = [
    ("orders", "customer_id", "customers"),
    ("order_items", "order_id", "orders"),
    ("order_items", "product_id", "products"),
    ("order_items", "seller_id", "sellers"),
    ("payments", "order_id", "orders"),
    ("reviews", "order_id", "orders"),
    ("products", "product_category_name", "category_translation"),
]

resultados_referencias = []

for origem, coluna, destino in relacoes:
    dados_origem = spark.table(f"olist_mvp.bronze.{origem}")

    chaves_destino = (
        spark.table(f"olist_mvp.bronze.{destino}")
        .select(coluna)
        .filter(~campo_ausente(coluna))
        .distinct()
    )

    referencias_ausentes = (
        dados_origem
        .filter(campo_ausente(coluna))
        .count()
    )

    # Conta linhas preenchidas sem correspondência no destino.
    sem_correspondencia = (
        dados_origem
        .filter(~campo_ausente(coluna))
        .join(chaves_destino, on=coluna, how="left_anti")
        .count()
    )

    resultados_referencias.append((
        origem,
        coluna,
        destino,
        referencias_ausentes,
        sem_correspondencia
    ))

display(
    spark.createDataFrame(
        resultados_referencias,
        """
        tabela_origem string,
        coluna string,
        tabela_destino string,
        referencias_ausentes long,
        referencias_sem_correspondencia long
        """
    )
)

tabela_origem,coluna,tabela_destino,referencias_ausentes,referencias_sem_correspondencia
orders,customer_id,customers,0,0
order_items,order_id,orders,0,0
order_items,product_id,products,0,0
order_items,seller_id,sellers,0,0
payments,order_id,orders,0,0
reviews,order_id,orders,0,0
products,product_category_name,category_translation,610,13


## Integridade referencial

Nas seis relações verificadas entre pedidos, clientes, itens,
produtos, vendedores, pagamentos e avaliações, todas as referências
estavam preenchidas e encontraram correspondência no destino.

Na relação entre produtos e traduções de categorias, foram
identificados 610 produtos sem categoria e 13 com categoria
preenchida sem correspondência na tabela de tradução.
Essas contagens representam produtos, não categorias distintas.

Os registros foram preservados. A ausência de tradução não impede
o uso da categoria original em português. Produtos sem categoria
podem ser agrupados como “Não informada” nas análises da Gold.

A verificação confirma a existência das referências no sentido
testado. Por exemplo, todos os itens apontarem para pedidos
existentes não garante que todos os pedidos possuam itens.

## Conclusão da camada Bronze

Foram carregadas oito tabelas Delta no esquema `olist_mvp.bronze`,
com as colunas de origem como texto e metadados de arquivo
e momento da ingestão.

Não foram identificadas duplicatas exatas. Das 47 colunas
de origem, 13 apresentaram ausências, concentradas em pedidos,
produtos e avaliações.

As chaves testadas estavam completas. Na tabela de avaliações,
`review_id` isoladamente apresentou repetições, enquanto a
combinação `review_id` e `order_id` foi única nesta carga.

O diagnóstico documenta limitações que precisam ser consideradas
no tratamento e nas análises. Nenhum registro foi removido
ou corrigido nesta camada.

A Silver realiza a padronização dos campos, a conversão de tipos
e as verificações de domínios, consistência temporal e valores
atípicos. A Gold aplica os critérios específicos de cada indicador.